In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import os

df = pd.read_csv('../data/raw/shipment-data.csv')
print("Shape:", df.shape)
df.head()

Shape: (10999, 12)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


In [2]:
# Drop Unnecessary Column
# ID column has no predictive value
df.drop(columns=['ID'], inplace=True)
print("Columns after dropping ID:", df.columns.tolist())

Columns after dropping ID: ['Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms', 'Reached.on.Time_Y.N']


In [3]:
# Check Data Types

print("Categorical columns:")
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(cat_cols)

print("\nNumerical columns:")
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(num_cols)

Categorical columns:
['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']

Numerical columns:
['Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 'Discount_offered', 'Weight_in_gms', 'Reached.on.Time_Y.N']


In [4]:
# Encode Categorical Columns
# Columns to encode
encode_cols = ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']

label_encoders = {}

for col in encode_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

df.head()
 

Warehouse_block: {'A': np.int64(0), 'B': np.int64(1), 'C': np.int64(2), 'D': np.int64(3), 'F': np.int64(4)}
Mode_of_Shipment: {'Flight': np.int64(0), 'Road': np.int64(1), 'Ship': np.int64(2)}
Product_importance: {'high': np.int64(0), 'low': np.int64(1), 'medium': np.int64(2)}
Gender: {'F': np.int64(0), 'M': np.int64(1)}


,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,3,0,4,2,177,3,1,0,44,1233,1
1,4,0,4,5,216,2,1,1,59,3088,1
2,0,0,2,2,183,4,1,1,48,3374,1
3,1,0,3,3,176,4,2,1,10,1177,1
4,2,0,2,2,184,3,2,0,46,2484,1


In [5]:
# Feature Engineering (Add New Features)

# High discount flag — from EDA we know >10% discount = almost always late
df['high_discount'] = (df['Discount_offered'] > 10).astype(int)

# Weight bucket — from EDA we know 2-4kg has highest delays
df['weight_bucket'] = pd.cut(df['Weight_in_gms'],
    bins=[0, 1000, 2000, 3000, 4000, 5000, 10000],
    labels=[0, 1, 2, 3, 4, 5]).astype(int)

# High call risk flag — 4+ calls = strong delay signal
df['high_call_risk'] = (df['Customer_care_calls'] >= 4).astype(int)

print("New features added:")
print(df[['high_discount', 'weight_bucket', 'high_call_risk']].value_counts().head(10))
print("\nNew shape:", df.shape)

New features added:
high_discount  weight_bucket  high_call_risk
0              1              1                 1939
               4              1                 1831
               5              1                 1727
               4              0                 1246
               5              0                 1156
1              1              1                  600
                              0                  522
               2              1                  464
               3              1                  423
                              0                  326
Name: count, dtype: int64

New shape: (10999, 14)


In [6]:
# Split Features and Target
X = df.drop(columns=['Reached.on.Time_Y.N'])
y = df['Reached.on.Time_Y.N']

print("Features (X):", X.shape)
print("Target (y):", y.shape)
print("\nFeature columns:", X.columns.tolist())

Features (X): (10999, 13)
Target (y): (10999,)

Feature columns: ['Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms', 'high_discount', 'weight_bucket', 'high_call_risk']


In [13]:
# Scale Numerical Features

# Columns to scale
scale_cols = ['Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product',
              'Prior_purchases', 'Discount_offered', 'Weight_in_gms']

scaler = StandardScaler()
X[scale_cols] = scaler.fit_transform(X[scale_cols])

print("Scaling done. Sample of scaled data:")
X[scale_cols].describe().round(2)

Scaling done. Sample of scaled data:


,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Discount_offered,Weight_in_gms
count,10999.00,10999.00,10999.00,10999.00,10999.00,10999.00
mean,0.00,-0.00,-0.00,-0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.80,-1.41,-2.38,-1.03,-0.76,-1.61
25%,-0.92,-0.70,-0.86,-0.37,-0.58,-1.10
50%,-0.05,0.01,0.08,-0.37,-0.39,0.31
75%,0.83,0.71,0.85,0.28,-0.21,0.87
max,2.58,1.42,2.08,4.22,3.19,2.58


In [7]:
# X = df.drop(columns=['Reached.on.Time_Y.N'])
y = df['Reached.on.Time_Y.N']

print("Features (X):", X.shape)
print("Target (y):", y.shape)
print("\nFeature columns:", X.columns.tolist())

Features (X): (10999, 13)
Target (y): (10999,)

Feature columns: ['Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms', 'high_discount', 'weight_bucket', 'high_call_risk']


In [8]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set   : {X_train.shape}")
print(f"Test set       : {X_test.shape}")
print(f"\nTrain target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

Training set   : (8799, 13)
Test set       : (2200, 13)

Train target distribution:
Reached.on.Time_Y.N
1    5250
0    3549
Name: count, dtype: int64

Test target distribution:
Reached.on.Time_Y.N
1    1313
0     887
Name: count, dtype: int64


In [9]:
# Save Processed Data

# Save splits
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Processed data saved to data/processed/")

Processed data saved to data/processed/


In [14]:
# Save Encoders and Scaler
os.makedirs('../models', exist_ok=True)

joblib.dump(label_encoders, '../models/label_encoders.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print("Saved:")
print("  models/label_encoders.pkl")
print("  models/scaler.pkl")


Saved:
  models/label_encoders.pkl
  models/scaler.pkl


In [15]:
# Final Check

print("=" * 50)
print("   PREPROCESSING COMPLETE")
print("=" * 50)
print(f"\n  Original shape   : (10999, 12)")
print(f"  After drop ID    : (10999, 11)")
print(f"  After engineering: {df.shape}")
print(f"\n  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"  y_test  : {y_test.shape}")
print(f"\n  Files saved to data/processed/")
print(f"  Encoders saved to models/")
print("\n  NEXT STEP: Run 03_modeling.ipynb")
print("=" * 50)

   PREPROCESSING COMPLETE

  Original shape   : (10999, 12)
  After drop ID    : (10999, 11)
  After engineering: (10999, 14)

  X_train : (8799, 13)
  X_test  : (2200, 13)
  y_train : (8799,)
  y_test  : (2200,)

  Files saved to data/processed/
  Encoders saved to models/

  NEXT STEP: Run 03_modeling.ipynb
